### Verb selection

In this notebook, we will filter through VTA verb list to assess which verbs have ambiguity (has more than 1 FST analysis)

#### FST-RUNTIME INSTALLATION

Please install fst-runtime (wrapper for FST parser). It requires Python 3.12
- `conda create --name cg3 python=3.12`
- `conda activate cg3`
- `pip install fst-runtime`



In [1]:
from fst_runtime.fst import Fst
import pandas as pd


In [2]:
fst_binary_filename = "../data/fst/ojibwe.att"

fst = Fst(fst_binary_filename)

In [3]:
# test parsing "waabam" (to see)
word = "waabam"
result = fst.up_analysis(wordform=word)
for analysis in result:
    print(analysis)

FstOutput(output_string='waabam+VTA+Imp+Sim+2SgSubj+3SgProxObj', path_weight=None, input_string='waabam')
FstOutput(output_string='waabam+VTA+Imp+Sim+2SgSubj+3PlProxObj', path_weight=None, input_string='waabam')


In [4]:
def load_from_csv(filename:str, column_name:str = "Form1Surface") -> list[str]:
    """ Read from a csv and return wordform list """
    try:
        output = pd.read_csv(filename)
        return output[column_name].tolist()
    except Exception as e:
        print("Error reading file ", filename, "Error =", e)
        


In [5]:
def filter_word_list(word_list:list, fst: Fst, min_fst_count: int = 2, verbose:bool=False) -> dict:
    """ Filter word list, to keep wordforms that have at least 2 fst analyses """
    output = dict() 
    
    max_row = -1 # max number of rows to process. Set to -1 for full set.
    count = 0
    for word_form in word_list[:max_row]:
        fst_analyses = list(fst.up_analysis(word_form))
        fst_outputs = [analysis.output_string for analysis in fst_analyses]         
        
        if len(fst_outputs) >= min_fst_count:
            
            if verbose:
                print("Ambiguity detected.")
                print("Word form =", word_form)
                print("Analyses =", fst_outputs)
                print("-"*10)
            
            # add to result
            output[word_form] = fst_outputs
            
            count += 1
            print("Count =", count, end="\r")
        
    
    print("\nCompleted. Count of unique wordforms =", len(output))
    return output

In [6]:
# ambiguity_list = filter_word_list(word_list=word_form_list, fst=fst, verbose=True)

In [7]:
def write_to_csv(ambiguity_list:dict, output_filename:str):
    """ Write the ambiguity list to csv file """
    # create pandas dataframe to hold output
    output_df = pd.DataFrame(
        {"wordform": ambiguity_list.keys(),
        "fst_analyses": ambiguity_list.values(),
        }
        )
    # output_df.head()

    # print("Writing ambiguity list to file =", output_filename)
    output_df.to_csv(output_filename, index=False)
    # print("Completed")

### Batch processing

In [8]:
input_filenames = [
    "../data/verbs/VTA_IND.csv",
    "../data/verbs/VTA_IMP.csv",
    "../data/verbs/VTA_CNJ.csv",
]

output_filenames = [
     "../data/verbs/VTA_IND_AMBIGUITY_LIST.csv",
     "../data/verbs/VTA_IMP_AMBIGUITY_LIST.csv",
     "../data/verbs/VTA_CNJ_AMBIGUITY_LIST.csv"
]


In [9]:
for input_filename, output_filename in zip(input_filenames, output_filenames):
    print("Reading from file =", input_filename)
    
    word_form_list = load_from_csv(filename=input_filename)                           # read wordforms from list

    print("Filtering words...")
    ambiguity_list = filter_word_list(word_list=word_form_list, fst=fst)              # filter to keep words with > 1 fst output

    print("Writing to file =", output_filename)
    write_to_csv(ambiguity_list=ambiguity_list, output_filename=output_filename)      # write to csv file
    print("Completed")
    
    print("-"*10)

Reading from file = ../data/verbs/VTA_IND.csv
Filtering words...
Count = 1123
Completed. Count of unique wordforms = 778
Writing to file = ../data/verbs/VTA_IND_AMBIGUITY_LIST.csv
Completed
----------
Reading from file = ../data/verbs/VTA_IMP.csv
Filtering words...
Count = 208
Completed. Count of unique wordforms = 94
Writing to file = ../data/verbs/VTA_IMP_AMBIGUITY_LIST.csv
Completed
----------
Reading from file = ../data/verbs/VTA_CNJ.csv
Filtering words...
Count = 1556
Completed. Count of unique wordforms = 801
Writing to file = ../data/verbs/VTA_CNJ_AMBIGUITY_LIST.csv
Completed
----------
